# 蚁群算法实验：校园无人配送路径规划（学生练习版）

本 Notebook 是学生练习版：第 2 部分的实验设置、代价建模和路网可视化已经完整给出，不作为考查内容；第 3 部分蚁群算法中的关键步骤被标记为 `TODO`，需要学生根据提示补全。

本实验使用 **蚁群算法 Ant Colony Optimization, ACO** 规划校园无人配送车路线。假设学校有一辆无人配送车，需要从校园快递站出发，把快递送到若干宿舍楼、教学楼或办公楼，最后返回快递站。

本实验的重点不是只找“距离最短”的路线，而是让学生理解真实路径规划中的综合代价：

1. 路段距离：越远代价越高。
2. 爬坡成本：坡度越大，能耗越高。
3. 路障成本：施工、封路、临时障碍会增加通行代价。
4. 拥堵成本：人流密集或车辆拥堵会增加时间代价。
5. 蚁群算法如何利用信息素和启发函数逐步搜索较优路线。

运行下面的程序后，可以观察算法从随机探索逐渐收敛到较优配送路线的过程。

## 1. 蚁群算法相关知识

蚁群算法是一种模仿蚂蚁觅食行为的群体智能优化算法。自然界中的蚂蚁会在走过的路径上留下信息素，后来的蚂蚁更倾向于选择信息素浓度高的路径。如果某条路径较短、较容易通行，蚂蚁往返更快，信息素也会更快积累，于是越来越多蚂蚁会选择这条路径。

在路径规划问题中，可以把每一只“人工蚂蚁”看作一次候选路线构造过程。蚂蚁从起点出发，每次根据两类信息选择下一个访问地点：

- **信息素 pheromone**：表示历史上这条边出现在好路线中的频率。信息素越高，越容易被选择。
- **启发信息 heuristic**：通常与代价成反比。边的综合代价越小，启发信息越大，越容易被选择。

蚂蚁从快递站出发，依次访问所有配送点，再返回快递站。每只蚂蚁都得到一条完整路线和总代价。每轮迭代结束后，算法会让信息素挥发，并让较优路线在对应边上增加信息素。

### 选择概率

假设蚂蚁当前在地点 `i`，还没有访问的候选地点为 `j`，则选择 `j` 的概率可写成：

`P(i, j) = [tau(i, j)^alpha * eta(i, j)^beta] / sum([tau(i, k)^alpha * eta(i, k)^beta])`

其中：

- `tau(i, j)` 是边 `(i, j)` 上的信息素。
- `eta(i, j)` 是边 `(i, j)` 的启发信息，本实验设置为 `1 / 综合代价`。
- `alpha` 控制信息素的重要程度。
- `beta` 控制启发信息的重要程度。

### 信息素更新

每轮迭代结束后，先让所有边的信息素按比例挥发：

`tau = (1 - evaporation_rate) * tau`

然后根据优秀路线增加信息素：

`tau(i, j) += Q / route_cost`

路线总代价越小，增加的信息素越多。这样可以让算法逐步偏向更好的路线。

## 2. 实验设置

本实验把校园抽象成若干配送点，每个点有二维坐标。快递站编号为 `0`，无人车必须从这里出发并最终回到这里。

综合代价由四部分组成：

`综合代价 = 距离 + 爬坡成本 + 路障成本 + 拥堵成本`

为了让模型更贴近校园配送，本实验做如下设置：

1. 每个地点有一个海拔高度，两个地点之间的上坡会产生额外能耗，下坡成本较低。
2. 部分道路存在施工、路障或临时管制，设置较高路障成本。
3. 部分道路经过食堂、主干道或教学区高峰路段，设置拥堵成本。
4. 算法目标是访问全部配送点并回到快递站，使总代价尽可能小。

这类问题与旅行商问题 TSP 相似，但本实验的边权不只是几何距离，而是包含多个现实因素的综合通行代价。

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt

random.seed(42)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

@dataclass(frozen=True)
class CampusLocation:
    """校园配送点数据结构。

    参数：
    - name：地点名称，例如“快递站”“图书馆”。
    - x, y：地点在校园平面图上的二维坐标，用于计算距离和绘图。
    - elevation：地点海拔高度，用于计算上坡或下坡成本。
    """
    name: str
    x: float
    y: float
    elevation: float

# 校园配送点列表。编号 0 是快递站，也是无人配送车的出发点和返回点。
locations = [
    CampusLocation("快递站", 0, 0, 18),
    CampusLocation("一号宿舍楼", 2, 8, 26),
    CampusLocation("二号宿舍楼", 6, 9, 31),
    CampusLocation("三号宿舍楼", 9, 5, 24),
    CampusLocation("第一教学楼", 4, 2, 22),
    CampusLocation("第二教学楼", 8, 1, 20),
    CampusLocation("图书馆", 5, 5, 28),
    CampusLocation("行政办公楼", 11, 8, 35),
    CampusLocation("实验楼", 12, 3, 23),
]

# 特殊道路因素，键使用无向边 (min_id, max_id)。
# obstacle 表示路障、施工、绕行等额外成本；congestion 表示拥堵成本。
road_factors: Dict[Tuple[int, int], Dict[str, float]] = {
    (0, 1): {"obstacle": 1.0, "congestion": 4.0},
    (0, 4): {"obstacle": 0.0, "congestion": 2.0},
    (1, 2): {"obstacle": 4.0, "congestion": 3.0},
    (1, 6): {"obstacle": 2.0, "congestion": 5.0},
    (2, 7): {"obstacle": 1.0, "congestion": 4.0},
    (3, 7): {"obstacle": 6.0, "congestion": 2.0},
    (4, 6): {"obstacle": 0.0, "congestion": 6.0},
    (4, 5): {"obstacle": 2.0, "congestion": 2.0},
    (5, 8): {"obstacle": 0.0, "congestion": 3.0},
    (6, 7): {"obstacle": 3.0, "congestion": 4.0},
    (6, 8): {"obstacle": 5.0, "congestion": 2.0},
}

def edge_key(i: int, j: int) -> Tuple[int, int]:
    """把两个地点编号转换成无向边键。

    参数：
    - i, j：两个校园地点的编号。

    返回：
    - `(较小编号, 较大编号)`，用于在 `road_factors` 中查询道路因素。
    """
    return (min(i, j), max(i, j))

def euclidean_distance(a: CampusLocation, b: CampusLocation) -> float:
    """计算两个地点在平面坐标中的直线距离。

    参数：
    - a, b：两个校园地点对象。

    返回：
    - 两点之间的欧氏距离。
    """
    return math.hypot(a.x - b.x, a.y - b.y)

def road_cost(i: int, j: int) -> float:
    """计算从地点 i 到地点 j 的综合通行代价。

    参数：
    - i：出发地点编号。
    - j：到达地点编号。

    返回：
    - 综合代价，包含距离、上坡成本、下坡成本、路障成本和拥堵成本。
    """
    a, b = locations[i], locations[j]
    distance = euclidean_distance(a, b)

    # elevation_diff > 0 表示从 i 到 j 是上坡；反之是下坡。
    elevation_diff = b.elevation - a.elevation
    uphill_cost = max(0, elevation_diff) * 0.45
    downhill_cost = max(0, -elevation_diff) * 0.10

    factors = road_factors.get(edge_key(i, j), {"obstacle": 0.0, "congestion": 0.0})
    obstacle_cost = factors["obstacle"]
    congestion_cost = factors["congestion"]

    return distance + uphill_cost + downhill_cost + obstacle_cost + congestion_cost

def build_cost_matrix() -> List[List[float]]:
    """构建综合代价矩阵。

    参数：
    - 无。函数直接使用全局变量 `locations` 和 `road_factors`。

    返回：
    - 二维列表 `matrix`，其中 `matrix[i][j]` 表示从地点 i 到地点 j 的综合代价。
    """
    n = len(locations)
    matrix = [[0.0 for _ in range(n)] for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = road_cost(i, j)
    return matrix

cost_matrix = build_cost_matrix()

print("校园配送点：")
for idx, loc in enumerate(locations):
    print(f"{idx}: {loc.name:6s} 坐标=({loc.x:>4.1f}, {loc.y:>4.1f}) 海拔={loc.elevation:>4.1f}")

In [ ]:
def show_cost_matrix(matrix: List[List[float]]) -> None:
    """打印综合代价矩阵。

    参数：
    - matrix：二维代价矩阵，通常由 `build_cost_matrix()` 生成。

    返回：
    - 无。函数只负责把矩阵以表格形式输出。
    """
    names = [loc.name for loc in locations]
    print("综合代价矩阵，行表示出发点，列表示到达点：")
    print(" " * 10 + " ".join(f"{i:^8d}" for i in range(len(names))))
    for i, row in enumerate(matrix):
        values = " ".join(f"{value:8.2f}" for value in row)
        print(f"{i:>2d} {names[i]:<6s} {values}")

show_cost_matrix(cost_matrix)

In [ ]:
def get_road_network_edges() -> List[Tuple[int, int]]:
    """生成完整路网边集合。

    参数：
    - 无。函数直接使用 `locations` 的长度生成所有地点对。

    返回：
    - 路网边列表，每条边形如 `(i, j)`。
    """
    edges = []
    for i in range(len(locations)):
        for j in range(i + 1, len(locations)):
            edges.append((i, j))
    return edges

road_network_edges = get_road_network_edges()

def draw_network_background(ax) -> None:
    """绘制浅灰色完整路网背景。

    参数：
    - ax：Matplotlib 坐标轴对象，表示要在哪张图上绘制。

    返回：
    - 无。函数直接在 `ax` 上画线。
    """
    for i, j in road_network_edges:
        a, b = locations[i], locations[j]
        ax.plot([a.x, b.x], [a.y, b.y], color="#c7c7c7", linewidth=0.8, alpha=0.35, zorder=1)

def draw_base_map(ax, title: str, show_network: bool = True) -> None:
    """绘制校园路网和配送点位底图。

    参数：
    - ax：Matplotlib 坐标轴对象。
    - title：当前子图标题。
    - show_network：是否绘制浅灰色完整路网，默认值为 True。

    返回：
    - scatter 对象，用于后续生成海拔颜色条。
    """
    if show_network:
        draw_network_background(ax)
    xs = [loc.x for loc in locations]
    ys = [loc.y for loc in locations]
    elevations = [loc.elevation for loc in locations]
    scatter = ax.scatter(xs, ys, c=elevations, cmap="YlGnBu", s=120, edgecolor="#333333", zorder=3)
    for idx, loc in enumerate(locations):
        ax.text(loc.x + 0.15, loc.y + 0.15, f"{idx} {loc.name}", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("校园平面 x 坐标")
    ax.set_ylabel("校园平面 y 坐标")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_aspect("equal", adjustable="box")
    return scatter

def draw_edge(ax, i: int, j: int, color: str, linewidth: float, linestyle: str = "-") -> None:
    """在图中绘制一条指定样式的路段。

    参数：
    - ax：Matplotlib 坐标轴对象。
    - i, j：路段两端地点编号。
    - color：线条颜色，例如红色表示路障，紫色表示拥堵。
    - linewidth：线条宽度，数值越大线越粗。
    - linestyle：线型，默认实线；例如 `"--"` 表示虚线。

    返回：
    - 无。函数直接在 `ax` 上画线。
    """
    a, b = locations[i], locations[j]
    ax.plot([a.x, b.x], [a.y, b.y], color=color, linewidth=linewidth, linestyle=linestyle, alpha=0.85, zorder=2)

def draw_uphill_arrow(ax, i: int, j: int) -> None:
    """绘制从低海拔指向高海拔的上坡箭头。

    参数：
    - ax：Matplotlib 坐标轴对象。
    - i, j：两个地点编号，函数内部会自动判断哪个地点更低。

    返回：
    - 无。函数直接在 `ax` 上绘制箭头和上坡高度差。
    """
    a, b = locations[i], locations[j]
    if a.elevation > b.elevation:
        i, j = j, i
        a, b = b, a
    ax.annotate(
        "",
        xy=(b.x, b.y),
        xytext=(a.x, a.y),
        arrowprops={"arrowstyle": "->", "color": "#2e7d32", "lw": 2.2, "alpha": 0.85},
        zorder=2,
    )
    mid_x = (a.x + b.x) / 2
    mid_y = (a.y + b.y) / 2
    ax.text(mid_x, mid_y, f"上坡+{b.elevation - a.elevation:.0f}m", color="#1b5e20", fontsize=8)

def get_typical_uphill_edges(limit: int = 8) -> List[Tuple[int, int, float]]:
    """选出坡度较明显的若干条上坡路段。

    参数：
    - limit：最多返回多少条上坡路段，默认值为 8。

    返回：
    - 列表，每个元素为 `(i, j, slope_score)`。
    - slope_score 表示坡度强弱，值越大说明单位距离内高度变化越明显。
    """
    uphill_edges = []
    for i in range(len(locations)):
        for j in range(i + 1, len(locations)):
            a, b = locations[i], locations[j]
            elevation_diff = abs(a.elevation - b.elevation)
            if elevation_diff >= 6:
                distance = euclidean_distance(a, b)
                slope_score = elevation_diff / distance
                uphill_edges.append((i, j, slope_score))
    uphill_edges.sort(key=lambda item: item[2], reverse=True)
    return uphill_edges[:limit]

def plot_full_road_network() -> None:
    """绘制完整校园路网图。

    参数：
    - 无。函数使用全局变量 `locations` 和 `road_network_edges`。

    返回：
    - 无。函数直接显示图像。
    """
    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    scatter = draw_base_map(ax, "校园完整路网图")
    ax.text(0.02, 0.02, f"路网边数量：{len(road_network_edges)}", transform=ax.transAxes, fontsize=10,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#999999"})
    fig.colorbar(scatter, ax=ax, shrink=0.82, label="海拔高度")
    plt.show()

def plot_road_factor_maps() -> None:
    """基于完整路网分别绘制上坡、路障和拥堵图。

    参数：
    - 无。函数使用全局变量 `locations`、`road_factors` 和 `road_network_edges`。

    返回：
    - 无。函数直接显示图像。
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), constrained_layout=True)

    scatter = draw_base_map(axes[0], "典型上坡路段")
    for i, j, _ in get_typical_uphill_edges():
        draw_uphill_arrow(axes[0], i, j)

    draw_base_map(axes[1], "路障 / 施工路段")
    for (i, j), factor in road_factors.items():
        obstacle = factor.get("obstacle", 0.0)
        if obstacle > 0:
            draw_edge(axes[1], i, j, color="#c62828", linewidth=1.2 + obstacle * 0.45)
            mid_x = (locations[i].x + locations[j].x) / 2
            mid_y = (locations[i].y + locations[j].y) / 2
            axes[1].text(mid_x, mid_y, f"障碍 {obstacle:.0f}", color="#8e0000", fontsize=8)

    draw_base_map(axes[2], "拥堵路段")
    for (i, j), factor in road_factors.items():
        congestion = factor.get("congestion", 0.0)
        if congestion > 0:
            draw_edge(axes[2], i, j, color="#6a1b9a", linewidth=1.2 + congestion * 0.35, linestyle="--")
            mid_x = (locations[i].x + locations[j].x) / 2
            mid_y = (locations[i].y + locations[j].y) / 2
            axes[2].text(mid_x, mid_y, f"拥堵 {congestion:.0f}", color="#4a148c", fontsize=8)

    fig.colorbar(scatter, ax=axes, shrink=0.82, label="海拔高度")
    plt.show()

def print_road_factor_summary() -> None:
    """打印上坡、路障和拥堵路段清单。

    参数：
    - 无。函数使用全局变量 `locations` 和 `road_factors`。

    返回：
    - 无。函数只负责打印文字说明。
    """
    print("典型上坡路段：")
    for i, j, _ in get_typical_uphill_edges():
        low, high = (i, j) if locations[i].elevation <= locations[j].elevation else (j, i)
        diff = locations[high].elevation - locations[low].elevation
        print(f"{locations[low].name} -> {locations[high].name}: 上坡 {diff:.0f}m")

    print("\n路障 / 施工路段：")
    for (i, j), factor in road_factors.items():
        if factor.get("obstacle", 0.0) > 0:
            print(f"{locations[i].name} - {locations[j].name}: 路障成本 {factor['obstacle']:.0f}")

    print("\n拥堵路段：")
    for (i, j), factor in road_factors.items():
        if factor.get("congestion", 0.0) > 0:
            print(f"{locations[i].name} - {locations[j].name}: 拥堵成本 {factor['congestion']:.0f}")

plot_full_road_network()
plot_road_factor_maps()
print_road_factor_summary()

## 3. 蚁群算法程序（学生补全）

下面的程序保留了蚁群算法的整体框架，但删除了若干关键实现。请按照 `TODO` 注释补全代码。

完成蚁群算法求解时，可以按下面顺序思考：

1. **理解已有输入**：第 2 部分已经构造好 `locations`、`road_factors` 和 `cost_matrix`，其中 `cost_matrix[i][j]` 表示从地点 `i` 到地点 `j` 的综合代价。
2. **计算路线总代价**：先确认 `route_cost(route)` 能把一条路线中相邻路段的代价累加起来。
3. **实现下一个地点选择**：在 `choose_next_node(current, unvisited)` 中，根据 `信息素^alpha * 启发信息^beta` 计算候选地点权重，再用轮盘赌方法选择下一个地点。
4. **构造单只蚂蚁路线**：在 `construct_route()` 中，从快递站出发，反复调用 `choose_next_node`，直到访问完所有地点，最后回到快递站。
5. **实现信息素挥发**：在 `evaporate_pheromone()` 中，让所有边的信息素乘以 `(1 - evaporation_rate)`，并设置一个很小的下限，避免信息素变成 0。
6. **实现信息素增加**：在 `deposit_pheromone(routes_with_costs)` 中，让较优路线按 `q / cost` 增加信息素，并额外强化历史最优路线。
7. **组织完整迭代过程**：在 `run()` 中，每一轮让多只蚂蚁构造路线、评价总代价、更新历史最优、选择较优路线、挥发并释放信息素。
8. **运行和观察结果**：补全代码后运行后续单元，观察最终路线、分段代价和收敛曲线。

建议先补全 `choose_next_node`，再补全 `construct_route`，最后完成信息素更新和 `run` 主循环。

In [ ]:
class AntColonyOptimizer:
    """蚁群算法优化器，用于搜索校园无人配送车的较优访问顺序。"""

    def __init__(
        self,
        cost_matrix: List[List[float]],
        num_ants: int = 30,
        num_iterations: int = 120,
        alpha: float = 1.0,
        beta: float = 3.0,
        evaporation_rate: float = 0.35,
        q: float = 100.0,
        elite_weight: float = 2.0,
        start_node: int = 0,
    ):
        """初始化蚁群算法参数。

        参数：
        - cost_matrix：综合代价矩阵，`cost_matrix[i][j]` 表示从 i 到 j 的代价。
        - num_ants：每轮迭代中的蚂蚁数量，越大搜索越充分。
        - num_iterations：最大迭代轮数，越大越可能找到更优路线。
        - alpha：信息素重要程度，越大越依赖历史优秀路线。
        - beta：启发信息重要程度，越大越偏向综合代价较小的路段。
        - evaporation_rate：信息素挥发率，越大旧信息遗忘越快。
        - q：信息素释放强度，越大优秀路线对后续搜索影响越强。
        - elite_weight：历史最优路线额外强化权重。
        - start_node：起点编号，本实验中 0 表示快递站。
        """
        self.cost_matrix = cost_matrix
        self.n = len(cost_matrix)
        self.num_ants = num_ants
        self.num_iterations = num_iterations
        self.alpha = alpha
        self.beta = beta
        self.evaporation_rate = evaporation_rate
        self.q = q
        self.elite_weight = elite_weight
        self.start_node = start_node
        self.pheromone = [[1.0 for _ in range(self.n)] for _ in range(self.n)]
        self.best_route: List[int] = []
        self.best_cost = float("inf")
        self.history: List[float] = []

    def route_cost(self, route: List[int]) -> float:
        """计算一条完整配送路线的总代价。

        参数：
        - route：地点编号序列，例如 `[0, 4, 1, 0]`。

        返回：
        - 路线上相邻地点代价之和。
        """
        return sum(self.cost_matrix[route[i]][route[i + 1]] for i in range(len(route) - 1))

    def choose_next_node(self, current: int, unvisited: set) -> int:
        """按照蚁群算法概率公式选择下一个访问地点。

        参数：
        - current：当前所在地点编号。
        - unvisited：尚未访问的地点编号集合。

        返回：
        - 被选中的下一个地点编号。
        """
        # TODO 1：把 unvisited 转换成候选地点列表 candidates。
        # TODO 2：对每个候选地点 node 计算：
        #         tau = 信息素 self.pheromone[current][node] 的 alpha 次方
        #         eta = 启发信息 (1 / self.cost_matrix[current][node]) 的 beta 次方
        #         weight = tau * eta
        # TODO 3：如果所有权重之和为 0，则随机选择一个候选地点。
        # TODO 4：使用轮盘赌方法，根据权重随机选择下一个地点。
        raise NotImplementedError("请补全 choose_next_node 方法")

    def construct_route(self) -> List[int]:
        """为一只蚂蚁构造一条完整配送路线。

        参数：
        - 无。函数使用对象中的 `start_node`、`cost_matrix` 和 `pheromone`。

        返回：
        - 完整路线，起点和终点都是快递站。
        """
        # TODO 1：创建路线 route，初始只包含起点 self.start_node。
        # TODO 2：创建 unvisited 集合，包含除起点以外的所有地点编号。
        # TODO 3：当 unvisited 不为空时，调用 choose_next_node 选择下一个地点。
        # TODO 4：把下一个地点加入 route，并从 unvisited 中删除。
        # TODO 5：所有地点访问完后，把起点加入 route 末尾，表示返回快递站。
        raise NotImplementedError("请补全 construct_route 方法")

    def evaporate_pheromone(self) -> None:
        """执行信息素挥发。

        参数：
        - 无。函数使用对象中的 `evaporation_rate`。

        返回：
        - 无。函数会直接更新 `self.pheromone`。
        """
        # TODO 1：遍历信息素矩阵中的每一条边。
        # TODO 2：让信息素乘以 (1 - self.evaporation_rate)。
        # TODO 3：使用 max(..., 1e-6) 给信息素设置下限，避免变成 0。
        raise NotImplementedError("请补全 evaporate_pheromone 方法")

    def deposit_pheromone(self, routes_with_costs: List[Tuple[List[int], float]]) -> None:
        """根据优秀路线增加信息素。

        参数：
        - routes_with_costs：列表，元素为 `(route, cost)`，表示路线及其总代价。

        返回：
        - 无。函数会直接更新 `self.pheromone`。
        """
        # TODO 1：遍历 routes_with_costs 中的每条路线及其总代价。
        # TODO 2：计算 delta = self.q / cost。
        # TODO 3：对路线中的每条边 a -> b 增加信息素；由于本实验道路视为双向，b -> a 也要增加。
        # TODO 4：如果 self.best_route 非空，使用 elite_weight 对历史最优路线额外增加信息素。
        raise NotImplementedError("请补全 deposit_pheromone 方法")

    def run(self) -> Tuple[List[int], float, List[float]]:
        """运行完整蚁群算法。

        参数：
        - 无。函数使用初始化时设置的所有算法参数。

        返回：
        - best_route：搜索到的历史最优路线。
        - best_cost：历史最优路线的总代价。
        - history：每轮迭代结束后的历史最优代价列表。
        """
        # TODO 1：进行 self.num_iterations 轮迭代。
        # TODO 2：每轮中让 self.num_ants 只蚂蚁分别构造路线，并计算路线总代价。
        # TODO 3：如果某条路线优于 self.best_cost，则更新 self.best_route 和 self.best_cost。
        # TODO 4：按总代价从小到大排序，选择前 1/3 的较优路线用于释放信息素。
        # TODO 5：调用 evaporate_pheromone 和 deposit_pheromone 更新信息素。
        # TODO 6：把每轮结束后的 self.best_cost 加入 self.history。
        # TODO 7：按示例格式打印部分迭代轮次的当前最优总代价。
        raise NotImplementedError("请补全 run 方法")

aco = AntColonyOptimizer(
    cost_matrix,
    num_ants=40,
    num_iterations=150,
    alpha=1.0,
    beta=3.0,
    evaporation_rate=0.35,
    q=120.0,
    elite_weight=2.5,
)

# 补全上面的 TODO 后，再运行这一行开始搜索。
best_route, best_cost, history = aco.run()

In [ ]:
def route_to_names(route: List[int]) -> List[str]:
    """把地点编号路线转换成地点名称路线。

    参数：
    - route：地点编号列表。

    返回：
    - 地点名称列表。
    """
    return [locations[i].name for i in route]

def print_route_detail(route: List[int]) -> None:
    """打印配送路线、总代价和每段路的代价。

    参数：
    - route：地点编号列表，通常使用蚁群算法输出的 `best_route`。

    返回：
    - 无。函数只负责打印结果。
    """
    print("最优配送路线：")
    print(" -> ".join(route_to_names(route)))
    print(f"\n总代价：{aco.route_cost(route):.2f}\n")
    print("分路段代价：")
    for i in range(len(route) - 1):
        a, b = route[i], route[i + 1]
        print(f"{locations[a].name:<8s} -> {locations[b].name:<8s}: {cost_matrix[a][b]:6.2f}")

print_route_detail(best_route)

In [ ]:
def plot_campus_route(route: List[int]) -> None:
    """在校园地图上绘制最终配送路线。

    参数：
    - route：地点编号列表，表示无人车访问顺序。

    返回：
    - 无。函数直接显示路线图。
    """
    plt.figure(figsize=(9, 7))

    xs = [loc.x for loc in locations]
    ys = [loc.y for loc in locations]
    plt.scatter(xs, ys, s=120, color="#2f80ed", zorder=3)

    for idx, loc in enumerate(locations):
        label = f"{idx} {loc.name}"
        plt.text(loc.x + 0.15, loc.y + 0.15, label, fontsize=10)

    route_x = [locations[i].x for i in route]
    route_y = [locations[i].y for i in route]
    plt.plot(route_x, route_y, color="#d62728", linewidth=2.5, marker="o", zorder=2)

    for step in range(len(route) - 1):
        a, b = route[step], route[step + 1]
        mid_x = (locations[a].x + locations[b].x) / 2
        mid_y = (locations[a].y + locations[b].y) / 2
        plt.text(mid_x, mid_y, str(step + 1), color="#111111", fontsize=9,
                 bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "edgecolor": "#999999"})

    plt.title("校园无人配送车较优路线")
    plt.xlabel("校园平面 x 坐标")
    plt.ylabel("校园平面 y 坐标")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.axis("equal")
    plt.show()

def plot_convergence(history: List[float]) -> None:
    """绘制蚁群算法收敛曲线。

    参数：
    - history：每轮迭代结束后的历史最优总代价列表。

    返回：
    - 无。函数直接显示折线图。
    """
    plt.figure(figsize=(8, 4.5))
    plt.plot(range(1, len(history) + 1), history, color="#00897b", linewidth=2)
    plt.title("蚁群算法收敛曲线")
    plt.xlabel("迭代轮数")
    plt.ylabel("历史最优总代价")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.show()

plot_campus_route(best_route)
plot_convergence(history)

## 4. 实验思考

完成实验后，可以尝试回答下面的问题：

1. 如果只考虑距离，不考虑爬坡、路障和拥堵，最终路线会发生什么变化？
2. 增大 `beta` 后，蚂蚁是否更倾向于选择局部代价较小的路段？
3. 增大 `evaporation_rate` 后，算法是否更容易忘记早期路线？收敛速度和稳定性有什么变化？
4. 如果某条道路正在施工，可以怎样修改 `road_factors` 让无人车自动绕行？
5. 本实验每个地点只访问一次。如果同一栋楼有多个快递量，如何把快递数量、车辆载重或电量约束加入模型？